In [0]:
from pyspark.sql.functions import (col,lit,to_date,trim, upper, lower )

In [0]:
%run "../includes/common_functions"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS f1.silver;

In [0]:
constructors_df = spark.read.table("f1.bronze.constructors")

In [0]:
constructors_renamed_df = constructors_df\
    .withColumnRenamed("constructorId", "constructor_id")\
    .withColumnRenamed("constructorRef", "constructor_ref")\
    .withColumnRenamed("name", "constructor_name")

In [0]:
constructors_renamed_df = constructors_renamed_df\
    .withColumn("constructor_ref", trim(col("constructor_ref")))\
    .withColumn("constructor_name", trim(col("constructor_name")))\
    .withColumn("nationality", trim(col("nationality")))\
    .withColumn("url", trim(col("url")))

In [0]:
constructors_final_df = add_ingestion_date(constructors_renamed_df) \
    .withColumn("environment", lit("production")) \
    .withColumn("file_date", to_date(lit("2026-05-07")))

In [0]:
constructors_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("f1.silver.constructors")

In [0]:
%sql

SELECT *
FROM f1.silver.constructors
LIMIT 10;